In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Sample data (e.g., 1D input vectors)
data = torch.randn(1000, 20)  # 1000 samples, 20 features

# Split data into training and validation sets
train_data = data[:800]
val_data = data[800:]

# Define the Autoencoder model
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()  # Assuming input data is normalized between 0 and 1
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Initialize the model, loss function, and optimizer
input_dim = 20
hidden_dim = 10
model = Autoencoder(input_dim, hidden_dim)

criterion = nn.MSELoss()  # Mean Squared Error for reconstruction loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(TensorDataset(train_data), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(val_data), batch_size=batch_size, shuffle=False)

# Training loop
epochs = 20
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        inputs = batch[0]
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation loop
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch[0]
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            val_loss += loss.item()
    
    # Print losses for each epoch
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")

# Save the trained model
torch.save(model.state_dict(), "autoencoder.pth")

print("Training complete!")


Epoch [1/20], Train Loss: 1.2397, Val Loss: 1.2647
Epoch [2/20], Train Loss: 1.2157, Val Loss: 1.2397
Epoch [3/20], Train Loss: 1.1913, Val Loss: 1.2118
Epoch [4/20], Train Loss: 1.1638, Val Loss: 1.1811
Epoch [5/20], Train Loss: 1.1335, Val Loss: 1.1472
Epoch [6/20], Train Loss: 1.1012, Val Loss: 1.1128
Epoch [7/20], Train Loss: 1.0692, Val Loss: 1.0796
Epoch [8/20], Train Loss: 1.0394, Val Loss: 1.0500
Epoch [9/20], Train Loss: 1.0132, Val Loss: 1.0252
Epoch [10/20], Train Loss: 0.9911, Val Loss: 1.0052
Epoch [11/20], Train Loss: 0.9731, Val Loss: 0.9892
Epoch [12/20], Train Loss: 0.9584, Val Loss: 0.9763
Epoch [13/20], Train Loss: 0.9463, Val Loss: 0.9658
Epoch [14/20], Train Loss: 0.9362, Val Loss: 0.9568
Epoch [15/20], Train Loss: 0.9276, Val Loss: 0.9492
Epoch [16/20], Train Loss: 0.9200, Val Loss: 0.9424
Epoch [17/20], Train Loss: 0.9132, Val Loss: 0.9362
Epoch [18/20], Train Loss: 0.9069, Val Loss: 0.9304
Epoch [19/20], Train Loss: 0.9010, Val Loss: 0.9251
Epoch [20/20], Train 

In [4]:
# Load the saved model
model.load_state_dict(torch.load("autoencoder.pth"))
model.eval()  # Set the model to evaluation mode

# Example single data point (ensure it matches the input dimensions)
single_data = torch.randn(1, 20)  # A single 20-dimensional input vector

# Forward pass to get the reconstructed output
with torch.no_grad():
    reconstructed = model(single_data)

# Calculate the reconstruction error
reconstruction_error = torch.nn.functional.mse_loss(reconstructed, single_data)

print("Original Data:", single_data.numpy())
print("Reconstructed Data:", reconstructed.numpy())
print("Reconstruction Error:", reconstruction_error.item())

Original Data: [[-0.12520416  0.0667019   1.5240245   1.9680303  -0.02595019  0.08254426
  -0.7307341   0.1558694   0.24827415 -0.4976751  -0.14002068 -0.64355975
  -1.0893574  -0.36248794 -1.9611458   0.5937581  -1.7008938   1.0239525
   0.94200236  0.00759142]]
Reconstructed Data: [[0.03762699 0.08962657 0.15685806 0.71131325 0.06784902 0.09972871
  0.12090911 0.17796816 0.14785373 0.06465396 0.23261787 0.01305958
  0.05196    0.1274215  0.02469924 0.17846659 0.14193931 0.35200626
  0.5467593  0.04589074]]
Reconstruction Error: 0.7385299205780029
